# 🧭 Notebook 2: Vector Clocks — capturing causality

A **vector clock** generalises Lamport: instead of one integer, each process keeps **one integer per process** in the system. So with 3 processes, every event has a vector like `[2, 5, 1]`.

Rules:

1. Local event in process `i` → increment `V[i]`.
2. Send → attach a copy of your vector.
3. Receive `V'` → for every `j`, `V[j] = max(V[j], V'[j])`, then increment `V[i]`.

Comparing two vectors `U` and `V`:
- **U ≤ V** if `U[i] ≤ V[i]` for all `i`. → `U` happened-before `V`.
- **U > V** symmetrically.
- Otherwise → **concurrent**.

This recovers the information Lamport throws away.

## Learning objectives
- Implement a vector clock.
- Detect concurrent events that Lamport mis-ordered.
- See how Dynamo-style systems use this to surface conflicts.

In [ ]:
def vle(u, v):  return all(a <= b for a, b in zip(u, v))
def vlt(u, v):  return vle(u, v) and u != v

def compare(u, v):
    if u == v:        return "equal"
    if vlt(u, v):     return "u → v (u happened before v)"
    if vlt(v, u):     return "v → u (v happened before u)"
    return "concurrent (no causal order)"

class VCProcess:
    def __init__(self, name, idx, n):
        self.name = name
        self.idx = idx
        self.vc = [0] * n
        self.history = []

    def local(self, label):
        self.vc[self.idx] += 1
        self.history.append((tuple(self.vc), f"{self.name}: {label}"))
        return tuple(self.vc)

    def send(self, target, label):
        self.vc[self.idx] += 1
        snap = list(self.vc)
        self.history.append((tuple(snap), f"{self.name}: send '{label}' -> {target.name}"))
        return snap, label

    def recv(self, vc_in, label):
        self.vc = [max(a, b) for a, b in zip(self.vc, vc_in)]
        self.vc[self.idx] += 1
        self.history.append((tuple(self.vc), f"{self.name}: recv '{label}'"))

A = VCProcess("A", 0, 3)
B = VCProcess("B", 1, 3)
C = VCProcess("C", 2, 3)

vc_a1 = A.local("write x=1")
m = A.send(B, "x=1"); B.recv(*m)
vc_b1 = B.local("write y=2")
m = B.send(C, "y=2"); C.recv(*m)
vc_c1 = C.local("write z=3")
vc_a2 = A.local("indep A1")
vc_c2 = C.local("indep C1")

for vc, ev in A.history + B.history + C.history:
    print(f"{vc}  {ev}")

In [ ]:
# Now ask the questions Lamport couldn't answer.
print("A1 vs C1:", compare(vc_a1, vc_c1))         # causally related (via B)
print("indep A1 vs indep C1:", compare(vc_a2, vc_c2))  # concurrent!
print("indep A1 vs C1 (write z):", compare(vc_a2, vc_c1))

## 🛒 Real use: Amazon Dynamo's shopping cart

When two replicas of your shopping cart receive different "add item" requests at the same time, vector clocks let the system **detect** the conflict. The user (or app) can then merge: union both carts. Without vector clocks, one update would silently overwrite the other.

| | Lamport | Vector clock |
|---|---|---|
| Storage per event | 1 int | N ints (N = #processes) |
| Total order | yes | no |
| Causal order | partial | full |
| Detects concurrency? | **no** | **yes** |

That's the trade: vector clocks cost more memory but tell you the truth about causality.